# Translate your own English↔Chinese conversation (the fine-tuned system, one cell)

After `train_eval_zh_kaggle.ipynb` has run once, this notebook needs **no training or evaluation**: run the two cells; the second one asks for your conversation.

**Settings:** Accelerator **GPU T4 x2**, Internet **On**.
**Input:** *Add Input → Notebook Output →* the notebook `train_eval_zh_kaggle` (it contains the adapter `zh_lora/`), or a dataset that contains it.

What happens for every message you type: the earlier messages of the conversation are used as context, the fine-tuned TowerInstruct-7B samples 6 candidate
translations, and context-aware COMET picks the best one (the paper's primary system, "contextual MBR re-ranking").

**Judge (optional):** *after* each translation Gemini grades it (GEMBA-MQM with context; score 0 = no errors found, more negative = worse). The judge only grades,
it never changes the translation. It needs the free key in *Add-ons → Secrets* as `GEMINI_API_KEY` (https://aistudio.google.com/apikey); without it the cell
simply runs without the judge.

In [ ]:
# 1. Setup (once per session)
import os, subprocess
os.chdir("/kaggle/working")
if not os.path.exists("cat"):
    subprocess.run(["git", "clone", "--depth", "1", "--filter=blob:none", "--sparse", "-b", "master", "https://github.com/namannamu18/context-aware-translation.git", "cat"], check=True)
os.chdir("/kaggle/working/cat")
subprocess.run(["git", "sparse-checkout", "set", "scripts"], check=True)
!pip install -q "transformers==4.46.3" "unbabel-comet==2.2.7" "peft>=0.13,<0.16" "sacrebleu>=2.4" "numpy<2" "protobuf<5" accelerate openai pandas
# If the next cell fails with an import error: Run -> Restart session, then run cell 2 again (skip cell 1).

In [ ]:
# 2. THE CELL: loads the system (a few minutes the first time), then asks for the conversation.
#    Type or paste messages starting with "en:" or "zh:" (several on one line are fine); "reset" = new conversation, "verbose" = also show the
#    greedy translations, "judge" = judge on/off, "quit" = stop and print the transcript.
import os, sys, glob
os.environ["USE_TF"] = "0"; os.environ["TRANSFORMERS_NO_TF"] = "1"
sys.path.insert(0, "/kaggle/working/cat/scripts")
hits = glob.glob("/kaggle/input/**/adapter_config.json", recursive=True)
assert hits, "Add the training notebook's output as input first (Add Input -> Notebook Output)."
ADAPTER = os.path.dirname(hits[0])
from translate_chat_pipeline import load_system, interactive_pipeline
from judge_chat import ChatJudge

USE_JUDGE = True                       # grade every translation with Gemini AFTER it was chosen (set False to skip)
JUDGE_MODEL = "gemini-2.5-flash"       # a current Flash model from AI Studio (names change; check the list there if the judge reports no score)
judge = None
if USE_JUDGE:
    try:
        from kaggle_secrets import UserSecretsClient
        os.environ["GEMINI_API_KEY"] = UserSecretsClient().get_secret("GEMINI_API_KEY")
    except Exception:
        pass                           # no secret attached: the judge is disabled below, with a message
    judge = ChatJudge.create(judge_model=JUDGE_MODEL)     # None without a key

pt = load_system(adapter=ADAPTER, n_candidates=6, judge=judge)    # base model + prompt format are read from the adapter; + COMET (+ judge)
interactive_pipeline(pt)